[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/metaflow-certified/notebooks/day-08-parameterized-flows.ipynb#scrollTo=10a2b3c4)

---
# Day 8 · Parameterized Flows and Configuration
**certified-journeys / metaflow-certified** · Practice · Reusable, configurable ML pipelines

> **Goal for today:** Write Metaflow flows that accept runtime parameters — scalars, JSON configs, and file contents — so one flow definition handles every environment without code changes.


In [ ]:
%pip install -q metaflow


## Step 1 · Basic Parameter — scalars and defaults

`Parameter` is a class attribute on your `FlowSpec`. It captures CLI arguments and makes them available as `self.<name>` in every step.

| Field | Purpose | Example |
|---|---|---|
| `name` | First positional arg — sets the CLI flag | `'learning_rate'` |
| `default` | Used when flag is omitted | `0.001` |
| `type` | Python type for parsing | `float`, `int`, `str`, `bool` |
| `help` | Shown in `--help` output | `'Learning rate for SGD'` |
| `required` | Raise error if missing | `True` |

CLI usage: `python flow.py run --learning_rate 0.01 --epochs 50`


In [ ]:
# Write a flow with scalar Parameters
basic_param_code = '''
from metaflow import FlowSpec, step, Parameter

class TrainingFlow(FlowSpec):
    """A training flow parameterized for any environment."""

    learning_rate = Parameter(
        'learning_rate',
        default=0.001,
        type=float,
        help='Learning rate for the optimizer'
    )

    epochs = Parameter(
        'epochs',
        default=10,
        type=int,
        help='Number of training epochs'
    )

    model_name = Parameter(
        'model_name',
        default='resnet18',
        type=str,
        help='Architecture to train'
    )

    dry_run = Parameter(
        'dry_run',
        default=False,
        type=bool,
        help='If True, skip actual training (useful for config validation)'
    )

    @step
    def start(self):
        print(f"Config received:")
        print(f"  model_name   = {self.model_name}")
        print(f"  learning_rate= {self.learning_rate}")
        print(f"  epochs       = {self.epochs}")
        print(f"  dry_run      = {self.dry_run}")
        self.next(self.train)

    @step
    def train(self):
        if self.dry_run:
            print("  [dry_run] Skipping actual training.")
            self.final_loss = None
        else:
            # Simulate training — in real life this calls your training framework
            simulated_loss = 2.3 * (0.95 ** self.epochs) * self.learning_rate * 1000
            self.final_loss = round(simulated_loss, 4)
            print(f"  Training complete. final_loss={self.final_loss}")
        self.next(self.end)

    @step
    def end(self):
        print(f"Run complete. model={self.model_name} loss={self.final_loss}")

if __name__ == "__main__":
    TrainingFlow()
'''

with open('training_flow.py', 'w') as f:
    f.write(basic_param_code)
print('training_flow.py written.')


In [ ]:
# Run with defaults
print("=== Run with defaults ===")
!python training_flow.py run --no-pylint 2>&1


In [ ]:
# Run with custom parameters — simulating a production config
print("=== Run with custom parameters ===")
!python training_flow.py run --no-pylint \
    --learning_rate 0.0001 \
    --epochs 50 \
    --model_name efficientnet_b0 2>&1


**What just happened?**
- Parameters are **class attributes** — Metaflow parses them from `sys.argv` before the first step runs.
- `self.learning_rate`, `self.epochs` etc. are available in *every step*, not just `start`.
- **Parameters are stored as run artifacts** — inspecting any past run shows you its exact config.
- `type=bool` with `--dry_run True` works; `--dry_run` (flag style) also sets it to `True`.


## Step 2 · JSONType — structured configuration objects

`JSONType` lets you pass a full dictionary or list as a single CLI argument. This is the idiomatic way to handle configuration objects that contain many keys.

```bash
# Pass a JSON dict on the CLI:
python flow.py run --config '{"batch_size": 64, "optimizer": "adam", "weight_decay": 1e-4}'

# Or from a file:
python flow.py run --config "$(cat config/prod.json)"
```


In [ ]:
json_param_code = '''
import json
from metaflow import FlowSpec, step, Parameter, JSONType

class ConfigurableFlow(FlowSpec):
    """Flow that accepts a full config dict as a single JSON parameter."""

    # JSONType parses a JSON string into a Python dict/list automatically
    config = Parameter(
        'config',
        default=\'{"batch_size": 32, "optimizer": "sgd", "weight_decay": 0.0001}\',
        type=JSONType,
        help='Training configuration as a JSON object'
    )

    # You can mix JSONType params with scalar params
    env = Parameter(
        'env',
        default='dev',
        type=str,
        help='Deployment environment: dev | staging | prod'
    )

    @step
    def start(self):
        # self.config is already a Python dict — no json.loads() needed
        print(f"Environment : {self.env}")
        print(f"Config type : {type(self.config).__name__}")
        print(f"Config keys : {list(self.config.keys())}")
        print(f"Batch size  : {self.config.get(\'batch_size\')}")
        print(f"Optimizer   : {self.config.get(\'optimizer\')}")
        self.next(self.apply_config)

    @step
    def apply_config(self):
        # Build an environment-specific config by overlaying env defaults
        env_overrides = {
            "dev":     {"batch_size": 8,   "debug": True},
            "staging": {"batch_size": 64,  "debug": False},
            "prod":    {"batch_size": 256, "debug": False, "mixed_precision": True},
        }
        base = dict(self.config)  # start from the passed config
        base.update(env_overrides.get(self.env, {}))  # overlay env defaults
        self.final_config = base
        print(f"Final config for {self.env}: {json.dumps(self.final_config, indent=2)}")
        self.next(self.end)

    @step
    def end(self):
        print(f"Ready to train with batch_size={self.final_config[\'batch_size\']}")

if __name__ == "__main__":
    ConfigurableFlow()
'''

with open('configurable_flow.py', 'w') as f:
    f.write(json_param_code)
print('configurable_flow.py written.')


In [ ]:
# Run with default (dev) config
print("=== Dev environment (default config) ===")
!python configurable_flow.py run --no-pylint 2>&1


In [ ]:
# Run with a prod-specific JSON config passed inline
print("=== Prod environment with custom config ===")
!python configurable_flow.py run --no-pylint \
    --env prod \
    --config '{"batch_size": 128, "optimizer": "adamw", "weight_decay": 0.01}' 2>&1


**What just happened?**
- `JSONType` automatically deserializes the CLI string into a Python `dict` — no `json.loads()` needed in your step code.
- **The `env` scalar parameter** overlays environment-specific defaults on top of the passed config.
- Using `--config "$(cat config/prod.json)"` in CI/CD is idiomatic — keep configs in version control, pass at runtime.
- `self.final_config` is saved as an artifact so you can audit exactly what config each historical run used.


## Step 3 · IncludeFile — passing file contents as a parameter

`IncludeFile` reads a local file at flow execution time and makes its contents available as a string artifact. This is useful for:
- SQL query files
- Prompt templates
- Small reference data files
- Config files you want stored alongside the run

The file is serialized into the run artifacts — so even if the original file changes, the run's copy is immutable.


In [ ]:
# First, create a sample SQL query file and a config file to include
import json

sql_query = """SELECT
    user_id,
    COUNT(*) AS event_count,
    MAX(event_time) AS last_seen
FROM events
WHERE event_date >= CURRENT_DATE - INTERVAL 30 DAYS
GROUP BY 1
HAVING event_count > 5
ORDER BY event_count DESC
LIMIT 10000
"""

feature_config = {
    "features": ["user_id", "event_count", "last_seen"],
    "target": "churn_label",
    "split_ratio": 0.8,
    "random_seed": 42
}

with open('query.sql', 'w') as f:
    f.write(sql_query)

with open('feature_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)

print('query.sql and feature_config.json created.')


In [ ]:
include_file_code = '''
import json
from metaflow import FlowSpec, step, Parameter, JSONType, IncludeFile

class FeatureEngineeringFlow(FlowSpec):
    """Flow that includes a SQL query file and a feature config file as parameters."""

    # IncludeFile reads the file at runtime and stores its contents as a string artifact
    query_sql = IncludeFile(
        'query_sql',
        default='query.sql',
        help='SQL file to run for feature extraction',
        is_text=True  # Decode bytes to string; False for binary files
    )

    feature_config = IncludeFile(
        'feature_config',
        default='feature_config.json',
        help='JSON file defining features and model config',
        is_text=True
    )

    env = Parameter(
        'env',
        default='dev',
        type=str,
        help='Target environment'
    )

    @step
    def start(self):
        # self.query_sql is the full file content as a string
        print("SQL query loaded:")
        print(self.query_sql[:150], "...")

        # Parse the JSON config
        self.parsed_config = json.loads(self.feature_config)
        print(f"\nFeature config: {self.parsed_config}")
        self.next(self.extract_features)

    @step
    def extract_features(self):
        # Simulate running the SQL query against a data warehouse
        # Production: run against DuckDB, BigQuery, Snowflake etc.
        features = self.parsed_config["features"]
        print(f"  Extracting features: {features}")
        print(f"  SQL hash: {hash(self.query_sql) & 0xFFFFFF:06x}")
        # Fake extracted data
        self.feature_matrix = {
            "rows": 9847,
            "cols": len(features),
            "env": self.env
        }
        self.next(self.end)

    @step
    def end(self):
        print(f"Features extracted: {self.feature_matrix}")
        print("The SQL and config are stored as immutable run artifacts.")

if __name__ == "__main__":
    FeatureEngineeringFlow()
'''

with open('feature_flow.py', 'w') as f:
    f.write(include_file_code)
print('feature_flow.py written.')


In [ ]:
!python feature_flow.py run --no-pylint 2>&1


**What just happened?**
- `IncludeFile` read `query.sql` at flow start time and stored its full text in `self.query_sql` as an artifact.
- **The file is snapshotted into the run** — if `query.sql` changes after this run, the historical run still has the exact SQL that was used.
- `is_text=True` decodes the file as UTF-8; set `is_text=False` for binary files (model weights, images).
- Override the default path at runtime: `python feature_flow.py run --query_sql prod_query.sql`.


## Step 4 · Parameter patterns for dev / staging / prod

A common production pattern is to keep per-environment JSON config files in your repo and select one at runtime. This avoids hard-coding environment logic inside the flow.

```bash
# Dev run — fast, small data
python flow.py run --config "$(cat configs/dev.json)"

# Staging run — full data, staging infra
python flow.py run --config "$(cat configs/staging.json)"

# Prod run via CI/CD — reads config from environment variable
python flow.py run --config "$PROD_CONFIG_JSON"
```


In [ ]:
# Create per-environment config files
import json, os

os.makedirs('configs', exist_ok=True)

configs = {
    'configs/dev.json': {
        "data_source": "local_sample",
        "sample_rows": 1000,
        "model": "logistic_regression",
        "output_bucket": "dev-ml-artifacts",
        "enable_monitoring": False
    },
    'configs/staging.json': {
        "data_source": "staging_warehouse",
        "sample_rows": 100000,
        "model": "xgboost",
        "output_bucket": "staging-ml-artifacts",
        "enable_monitoring": True
    },
    'configs/prod.json': {
        "data_source": "prod_warehouse",
        "sample_rows": -1,  # -1 means all rows
        "model": "xgboost",
        "output_bucket": "prod-ml-artifacts",
        "enable_monitoring": True
    }
}

for path, cfg in configs.items():
    with open(path, 'w') as f:
        json.dump(cfg, f, indent=2)
    print(f'Created {path}')


In [ ]:
multi_env_code = '''
import json
from metaflow import FlowSpec, step, Parameter, JSONType

class MultiEnvFlow(FlowSpec):
    """Single flow definition that runs across dev/staging/prod via JSON config."""

    config = Parameter(
        'config',
        default=\'{"data_source": "local_sample", "sample_rows": 1000, "model": "logistic_regression", "output_bucket": "dev-ml-artifacts", "enable_monitoring": false}\',
        type=JSONType,
        help='Full environment config as JSON'
    )

    @step
    def start(self):
        print(f"Environment config:")
        for k, v in self.config.items():
            print(f"  {k:25s} = {v}")
        self.next(self.load_data)

    @step
    def load_data(self):
        src = self.config["data_source"]
        rows = self.config["sample_rows"]
        if rows == -1:
            print(f"  Loading ALL rows from {src}")
            self.row_count = 5_000_000  # simulated
        else:
            print(f"  Loading {rows:,} rows from {src}")
            self.row_count = rows
        self.next(self.train)

    @step
    def train(self):
        model = self.config["model"]
        print(f"  Training {model} on {self.row_count:,} rows")
        # Simulate model artifact path
        bucket = self.config["output_bucket"]
        self.model_artifact_path = f"s3://{bucket}/models/run_latest.pkl"
        print(f"  Model saved to: {self.model_artifact_path}")
        self.next(self.end)

    @step
    def end(self):
        if self.config["enable_monitoring"]:
            print("  Monitoring enabled — emitting metrics to dashboard")
        print(f"Run complete. artifact={self.model_artifact_path}")

if __name__ == "__main__":
    MultiEnvFlow()
'''

with open('multi_env_flow.py', 'w') as f:
    f.write(multi_env_code)
print('multi_env_flow.py written.')


In [ ]:
import json

# Simulate the "$(cat configs/staging.json)" pattern in Python
with open('configs/staging.json') as f:
    staging_config_str = f.read().replace('\n', ' ')  # single-line for CLI

print("=== Staging environment ===")
import subprocess
result = subprocess.run(
    ['python', 'multi_env_flow.py', 'run', '--no-pylint', '--config', staging_config_str],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print(result.stderr[-1000:])


**What just happened?**
- One flow definition ran against the staging config without any code changes — only the `--config` flag differed.
- **`enable_monitoring`** toggled behavior based on config — no `if env == 'staging'` checks in the flow logic.
- In CI/CD, inject `$PROD_CONFIG_JSON` as an environment variable to parameterize without file access.
- Every past run has its exact config stored as an artifact — fully auditable.


## Step 5 · Parameter validation and required parameters

For safety, validate parameter values at the start of the flow before any expensive work begins. Metaflow does not provide built-in schema validation on `JSONType` — implement it yourself in `start`.


In [ ]:
validation_code = '''
from metaflow import FlowSpec, step, Parameter, JSONType

REQUIRED_CONFIG_KEYS = {"data_source", "model", "output_bucket"}
VALID_MODELS = {"logistic_regression", "xgboost", "random_forest", "lightgbm"}

class ValidatedFlow(FlowSpec):
    """Flow with upfront parameter validation to catch bad configs early."""

    config = Parameter(
        'config',
        default=\'{"data_source": "local", "model": "xgboost", "output_bucket": "dev-bucket"}\',
        type=JSONType
    )

    @step
    def start(self):
        # Validate required keys
        missing = REQUIRED_CONFIG_KEYS - set(self.config.keys())
        if missing:
            raise ValueError(f"Config is missing required keys: {missing}")

        # Validate enum values
        if self.config["model"] not in VALID_MODELS:
            raise ValueError(
                f"Unknown model \'{self.config[\'model\']}\'. "
                f"Must be one of: {VALID_MODELS}"
            )

        print("Config validation passed.")
        print(f"  model={self.config[\'model\']} bucket={self.config[\'output_bucket\']}")
        self.next(self.end)

    @step
    def end(self):
        print("Flow complete.")

if __name__ == "__main__":
    ValidatedFlow()
'''

with open('validated_flow.py', 'w') as f:
    f.write(validation_code)

# Run with a valid config
print("=== Valid config ===")
!python validated_flow.py run --no-pylint 2>&1


In [ ]:
# Run with an invalid config — observe the early failure
print("=== Invalid model name ===")
!python validated_flow.py run --no-pylint \
    --config '{"data_source": "prod", "model": "transformer", "output_bucket": "prod-bucket"}' 2>&1


**What just happened?**
- Validation runs in `start` — the cheapest possible step — so a bad config fails fast before any data loading.
- **Raise `ValueError` with a descriptive message** — Metaflow marks the run as failed and the message appears in logs.
- For complex configs, consider using `pydantic` for schema validation inside `start` — install it in the install cell.
- Required-parameter enforcement: set `required=True` on any `Parameter` that has no safe default.


In [ ]:
# Challenge: Build a HyperparamSearchFlow
#
# Create a flow that:
#   1. Takes a Parameter 'search_space' of JSONType with this default:
#      {"learning_rates": [0.01, 0.001, 0.0001], "batch_sizes": [32, 64]}
#
#   2. In 'start': validate that 'learning_rates' and 'batch_sizes' keys exist
#      and that both are non-empty lists
#
#   3. In 'expand': create self.trials — a list of dicts, one per
#      (learning_rate, batch_size) combination
#      e.g. [{"lr": 0.01, "bs": 32}, {"lr": 0.01, "bs": 64}, ...]
#
#   4. In 'end': print the number of trials and the full trial list
#
# Scaffold:
challenge_code = '''
from metaflow import FlowSpec, step, Parameter, JSONType

class HyperparamSearchFlow(FlowSpec):

    search_space = Parameter(
        'search_space',
        default=\'{"learning_rates": [0.01, 0.001, 0.0001], "batch_sizes": [32, 64]}\',
        type=JSONType,
        help='Hyperparameter search space'
    )

    @step
    def start(self):
        # TODO: validate that learning_rates and batch_sizes exist and are non-empty lists
        self.next(self.expand)

    @step
    def expand(self):
        # TODO: build self.trials as a list of {"lr": ..., "bs": ...} dicts
        self.trials = []
        self.next(self.end)

    @step
    def end(self):
        # TODO: print the trial count and the full list
        pass

if __name__ == "__main__":
    HyperparamSearchFlow()
'''

with open('hyperparam_flow.py', 'w') as f:
    f.write(challenge_code)
print('hyperparam_flow.py scaffold written — implement the TODOs, then run:')
print('  !python hyperparam_flow.py run --no-pylint')
print()
print('Bonus: also pass a custom search_space on the CLI, e.g.:')
print('  --search_space \'{"learning_rates": [0.1, 0.01], "batch_sizes": [16, 32, 128]}\'')


---
## Day 8 key concepts recap

| Concept | What to remember |
|---|---|
| `Parameter('name', default=..., type=...)` | Scalar CLI arg — available as `self.name` in every step |
| `JSONType` | Parses a JSON string into a Python dict/list automatically |
| `IncludeFile(...)` | Reads a file at run time; stores contents as an immutable artifact |
| Validate in `start` | Fail fast — check required keys and valid values before heavy steps |
| One flow, many envs | Pass `--config "$(cat configs/prod.json)"` to reuse one flow everywhere |
| Artifacts = audit trail | Every run records its exact parameter values — fully reproducible |

> **Tip:** Parameters make flows reusable — one flow definition can handle dev, staging, and prod configurations.

---
## What's next
**Day 9** → Debugging and the Client API: inspect past runs, fetch artifacts, resume failed flows, and generate visual run cards.

Mark Day 8 complete in your [tracker](../index.html).
